# 표 4 완성 — 신뢰구간 + AutoAttack 열 (누락분 측정)
세 부분으로 표 4의 빈칸을 채운다:
1. **셀 A (GPU 불필요):** blur_edge 캐시(CIFAR-10/Tiny/ImageNet)에서 FGSM/PGD/C&W의
   특징별 Overall AUROC에 **bootstrap 95% CI** 부여.
2. **셀 B (GPU):** CIFAR-100/SVHN을 torchvision으로 적재→224→FGSM/PGD/C&W/AutoAttack 생성→
   특징별 AUROC[CI] (AutoAttack 열 포함).
3. **셀 C (GPU):** CIFAR-10/ImageNet/Tiny의 **AutoAttack 열**을 정상 이미지로부터 생성·측정.
AutoAttack은 autoattack 패키지 부재 환경 기준 **APGD-CE(50스텝) 대체**(열 이름에 명시).

In [ ]:
import os, sys, subprocess, pickle, io as _io, glob
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.models as models, torchvision.transforms as T
from torchvision.models import ResNet50_Weights
from torchvision.transforms.functional import gaussian_blur
from PIL import Image as _Image
from sklearn.metrics import roc_auc_score
SEED=42; device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(SEED); torch.manual_seed(SEED)
def _find(name, ftype='f', maxdepth=8):
    roots=['/home','/root','/workspace',os.path.expanduser('~'),'.','..','../..','.']
    res=[]
    for r in roots:
        if not os.path.exists(r): continue
        try:
            out=subprocess.run(['find',r,'-maxdepth',str(maxdepth),'-type',ftype,'-name',name],
                               capture_output=True,text=True,timeout=20).stdout.strip()
            if out: res+=[p for p in out.split('\n') if p]
        except: pass
    return sorted(set(res))
CIFAR_MEAN=[0.4914,0.4822,0.4465]; CIFAR_STD=[0.2470,0.2435,0.2616]
IMGNET_MEAN=[0.485,0.456,0.406]; IMGNET_STD=[0.229,0.224,0.225]
def make_pp(ds):
    m,s=(CIFAR_MEAN,CIFAR_STD) if 'CIFAR' in ds else (IMGNET_MEAN,IMGNET_STD)
    mean=torch.tensor(m).view(1,3,1,1).to(device); std=torch.tensor(s).view(1,3,1,1).to(device)
    return lambda x:(x/255.0-mean)/std
def load_backbone(ds):
    cfg={'CIFAR-10':('resnet50_cifar10_finetuned.pt',10),'CIFAR-100':('resnet50_cifar100_finetuned.pt',100),
         'SVHN':('resnet50_svhn_finetuned.pt',10),'TinyImageNet':('resnet50_tinyimagenet_finetuned.pt',200)}
    if ds in cfg:
        ck=(_find(cfg[ds][0]) or [None])[0]; m=models.resnet50(weights=None); m.fc=nn.Linear(2048,cfg[ds][1])
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict']); nc=cfg[ds][1]
    else:
        m=models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2); nc=1000
    return m.to(device).eval(), nc
def gb(x,s):
    k=int(2*np.ceil(3*s)+1); k=k+1 if k%2==0 else k
    return gaussian_blur(x,kernel_size=k,sigma=s)
def to224(img):
    if img.dim()==3: img=img.unsqueeze(0)
    img=img.float()
    if img.shape[-1]!=224: img=F.interpolate(img,size=(224,224),mode='bicubic',align_corners=False)
    return img.clamp(0,255)
def jpeg(x,q=75):
    a=x.detach().squeeze(0).permute(1,2,0).clamp(0,255).byte().cpu().numpy()
    b=_io.BytesIO(); _Image.fromarray(a).save(b,format='JPEG',quality=int(q)); b.seek(0)
    return torch.from_numpy(np.array(_Image.open(b).convert('RGB'))).float().permute(2,0,1).unsqueeze(0).to(x.device)
def jpeg_batch(x): return torch.cat([jpeg(x[i:i+1]) for i in range(x.shape[0])],0)
def median3(x):
    xx=F.pad(x,(1,1,1,1),mode='reflect'); p=xx.unfold(2,3,1).unfold(3,3,1)
    return p.contiguous().view(*p.shape[:4],9).median(dim=-1).values
def feat_hfe(x): return ((x-gb(x,0.5)).abs().flatten(1).mean(1)/255.0).cpu().numpy()
def feat_gl(x,bb,pp,sig=1.0):
    with torch.no_grad(): p0=F.softmax(bb(pp(x)),1); p1=F.softmax(bb(pp(gb(x,sig))),1)
    return (p0-p1).abs().sum(1).cpu().numpy()
def feat_predl1(x,bb,pp):
    with torch.no_grad():
        p0=F.softmax(bb(pp(x)),1); sq=median3(jpeg_batch(x)).clamp(0,255); p1=F.softmax(bb(pp(sq)),1)
    return (p0-p1).abs().sum(1).cpu().numpy()
def auc_ci(neg,pos,B=1000,seed=SEED):
    neg=np.asarray(neg);pos=np.asarray(pos)
    base=roc_auc_score(np.r_[np.zeros(len(neg)),np.ones(len(pos))],np.r_[neg,pos])
    rng=np.random.RandomState(seed); b=[]
    for _ in range(B):
        nb=neg[rng.randint(0,len(neg),len(neg))]; pb=pos[rng.randint(0,len(pos),len(pos))]
        b.append(roc_auc_score(np.r_[np.zeros(len(nb)),np.ones(len(pb))],np.r_[nb,pb]))
    return (base,*np.percentile(b,[2.5,97.5]))
# 공격 (0..255)
def fgsm(x,y,bb,pp,eps=8.0):
    x=x.clone().requires_grad_(True); loss=F.cross_entropy(bb(pp(x)),y); g=torch.autograd.grad(loss,x)[0]
    return (x+eps*g.sign()).clamp(0,255).detach()
def pgd(x,y,bb,pp,eps=8.0,a=2.0,steps=20):
    xa=(x+torch.empty_like(x).uniform_(-eps,eps)).clamp(0,255).detach()
    for _ in range(steps):
        xa.requires_grad_(True); loss=F.cross_entropy(bb(pp(xa)),y); g=torch.autograd.grad(loss,xa)[0]
        xa=(xa+a*g.sign()).clamp(x-eps,x+eps).clamp(0,255).detach()
    return xa
def cw(x,y,bb,pp,c=1.0,steps=100,lr=0.01):
    xn=(x/255.).clamp(1e-4,1-1e-4); w=torch.atanh(xn*2-1).clone().requires_grad_(True); opt=torch.optim.Adam([w],lr=lr)
    for _ in range(steps):
        xa=(torch.tanh(w)+1)/2*255.; lo=bb(pp(xa)); zt=lo.gather(1,y.view(-1,1)).squeeze(1)
        oth=lo.clone(); oth.scatter_(1,y.view(-1,1),-1e9)
        loss=((xa-x)/255.).pow(2).flatten(1).sum(1)+c*torch.clamp(zt-oth.max(1).values,min=0.)
        opt.zero_grad(); loss.sum().backward(); opt.step()
    return ((torch.tanh(w)+1)/2*255.).detach()
def apgd_ce(x,y,bb,pp,eps=8.0,steps=50,a=2.0):  # AutoAttack 대체(APGD-CE)
    xa=(x+torch.empty_like(x).uniform_(-eps,eps)).clamp(0,255).detach()
    for _ in range(steps):
        xa.requires_grad_(True); loss=F.cross_entropy(bb(pp(xa)),y); g=torch.autograd.grad(loss,xa)[0]
        xa=(xa+a*g.sign()).clamp(x-eps,x+eps).clamp(0,255).detach()
    return xa
ATTACKS={'FGSM':fgsm,'PGD':pgd,'C&W':cw,'AutoAttack(APGD)':apgd_ce}
print('header ready; device=',device)

In [ ]:
# ── 셀 A: 캐시에서 FGSM/PGD/C&W CI (CIFAR-10/Tiny/ImageNet) ──
DSK={'CIFAR-10':'features_blur_edge_CIFAR-10.pkl','TinyImageNet':'features_blur_edge_TinyImageNet.pkl',
     'ImageNet':'features_blur_edge_ImageNet_eps8.pkl'}
GLKEY={'CIFAR-10':'gauss_l1_s0p5','TinyImageNet':'gauss_l1_s0p5','ImageNet':'gauss_l1_s1p0'}
FK=[('HF-Energy σ=0.5','hf_energy_s0p5'),('GaussianL1','__gl__'),('PredL1','predl1_jpeg')]
def feat_auc_cache(fd,key,atk=None):
    lab=np.array(fd['labels']); atks=np.array(fd['attacks']); a=np.array(fd[key],float)
    ci=np.where(lab==0)[0]; rng=np.random.RandomState(SEED); rng.shuffle(ci); half=ci[:len(ci)//2]
    mu=a[half].mean(); sd=a[half].std()+1e-8; anom=np.abs((a-mu)/sd)
    keep=np.array([(i in set(ci[len(ci)//2:].tolist())) or (lab[i]==1) for i in range(len(lab))])
    if atk: keep=keep&((atks=='clean')|(atks==atk))
    return auc_ci(anom[keep&(lab==0)], anom[keep&(lab==1)])
for ds,fn in DSK.items():
    c=[p for p in _find(fn) if 'blur_edge_results' in p] or _find(fn)
    if not c: print('skip(캐시없음)',ds); continue
    fd=pickle.load(open(c[0],'rb')); atks=sorted(set(a for a in fd['attacks'] if a!='clean'))
    print(f"\n[{ds}]  (cache)  특징별 Overall AUROC[95%CI]")
    for nm,k in FK:
        kk=GLKEY[ds] if k=='__gl__' else k
        if kk not in fd: print(f"  {nm:<16} (키없음 {kk})"); continue
        b,lo,hi=feat_auc_cache(fd,kk)
        print(f"  {nm:<16}{b:.3f} [{lo:.3f}, {hi:.3f}]")

In [ ]:
# ── 셀 B: CIFAR-100/SVHN 전체 (FGSM/PGD/C&W/AutoAttack) + CI (GPU) ──
ROOTS=['./cifar_data','./cifar10_data','./data','./cifar_data','./data']
SVHN_ROOTS=['./svhn_data','./data','./svhn_data']
def first_exist(c):
    for x in c:
        if os.path.exists(x): return x
    return c[0]
def load_ds(name,n=500):
    tf=T.ToTensor()
    if name=='CIFAR-100': d=torchvision.datasets.CIFAR100(first_exist(ROOTS),train=False,download=False,transform=tf)
    else: d=torchvision.datasets.SVHN(first_exist(SVHN_ROOTS),split='test',download=False,transform=tf)
    rng=np.random.RandomState(SEED); idx=rng.permutation(len(d))[:n]
    return torch.stack([d[i][0] for i in idx])*255.0, torch.tensor([int(d[i][1]) for i in idx])
def per_feat(clean224, adv_dict, bb, pp, glsig):
    def fa(x):
        H=[];G=[];P=[]
        for i in range(0,len(x),64):
            b=x[i:i+64].to(device); H.append(feat_hfe(b)); G.append(feat_gl(b,bb,pp,glsig)); P.append(feat_predl1(b,bb,pp))
        return np.concatenate(H),np.concatenate(G),np.concatenate(P)
    Hc,Gc,Pc=fa(clean224); res={}
    feats_adv={k:fa(v) for k,v in adv_dict.items()}
    for fi,(fname) in enumerate(['HF-Energy σ=0.5','GaussianL1','PredL1']):
        c=[Hc,Gc,Pc][fi]; mu=c.mean();sd=c.std()+1e-8; row={}
        alladv=[]
        for atk,(Ha,Ga,Pa) in feats_adv.items():
            a=[Ha,Ga,Pa][fi]; row[atk]=auc_ci(np.abs((c-mu)/sd),np.abs((a-mu)/sd)); alladv.append(a)
        row['Overall']=auc_ci(np.abs((c-mu)/sd),np.abs((np.concatenate(alladv)-mu)/sd)); res[fname]=row
    return res
for name in ['CIFAR-100','SVHN']:
    try: imgs,labs=load_ds(name,500)
    except Exception as e: print('skip',name,e); continue
    bb,nc=load_backbone(name); pp=make_pp(name)
    X=torch.cat([to224(imgs[i]) for i in range(len(imgs))],0); Y=labs
    with torch.no_grad():
        keep=torch.cat([(bb(pp(X[i:i+64].to(device))).argmax(1).cpu()==Y[i:i+64]) for i in range(0,len(X),64)])
    X=X[keep][:400]; Y=Y[keep][:400]
    adv={}
    for atk,fn in ATTACKS.items():
        outs=[]
        for i in range(0,len(X),64):
            b=X[i:i+64].to(device); yb=Y[i:i+64].to(device); outs.append(fn(b,yb,bb,pp).cpu())
        adv[atk]=torch.cat(outs,0)
    res=per_feat(X,adv,bb,pp,1.0)
    print(f"\n[{name}]  특징별 AUROC[95%CI]  (정확분류 {len(X)}장)")
    for fname,row in res.items():
        print(f"  {fname}")
        for atk in ['FGSM','PGD','C&W','AutoAttack(APGD)','Overall']:
            b,lo,hi=row[atk]; print(f"    {atk:<18}{b:.3f} [{lo:.3f}, {hi:.3f}]")

In [ ]:
# ── 셀 C: AutoAttack 열 — CIFAR-10/ImageNet/Tiny (정상→AutoAttack 생성) + CI (GPU) ──
def find_mixed():
    out={}
    for p in _find('mixed_dataset.pkl'):
        pl=p.lower()
        if 'cifar10' in pl or 'cifar_10' in pl: out['CIFAR-10']=p
        elif 'imagenet' in pl and 'eps8' in pl: out['ImageNet']=p
    return out
def tiny_val_images(n=400):
    base=( _find('val_annotations.txt') or [None])[0]
    if base is None: return None
    vdir=os.path.dirname(base); files=sorted(glob.glob(os.path.join(vdir,'images','*.JPEG')))[:n]
    if not files: files=sorted(glob.glob(os.path.join(vdir,'*.JPEG')))[:n]
    ims=[]
    for f in files:
        im=_Image.open(f).convert('RGB'); ims.append(torch.from_numpy(np.array(im)).float().permute(2,0,1))
    return ims
MX=find_mixed(); SRC={}
for ds in ['CIFAR-10','ImageNet']:
    if ds in MX:
        mixed=pickle.load(open(MX[ds],'rb')); SRC[ds]=[im for (im,lb,atk) in mixed if atk=='clean'][:400]
ti=tiny_val_images(400)
if ti is not None: SRC['TinyImageNet']=ti
for ds,imgs in SRC.items():
    bb,nc=load_backbone(ds if ds!='ImageNet' else 'ImageNet'); pp=make_pp(ds)
    X=torch.cat([to224(im) for im in imgs],0)
    with torch.no_grad():
        Y=torch.cat([bb(pp(X[i:i+64].to(device))).argmax(1).cpu() for i in range(0,len(X),64)])
    aa=[]
    for i in range(0,len(X),64):
        b=X[i:i+64].to(device); yb=Y[i:i+64].to(device); aa.append(apgd_ce(b,yb,bb,pp).cpu())
    Xa=torch.cat(aa,0); glsig=0.5 if ds=='CIFAR-10' else 1.0
    def fa(x):
        H=[];G=[];P=[]
        for i in range(0,len(x),64):
            b=x[i:i+64].to(device); H.append(feat_hfe(b));G.append(feat_gl(b,bb,pp,glsig));P.append(feat_predl1(b,bb,pp))
        return np.concatenate(H),np.concatenate(G),np.concatenate(P)
    Hc,Gc,Pc=fa(X); Ha,Ga,Pa=fa(Xa)
    print(f"\n[{ds}]  AutoAttack(APGD) 열  AUROC[95%CI]")
    for nm,c,a in [('HF-Energy σ=0.5',Hc,Ha),('GaussianL1',Gc,Ga),('PredL1',Pc,Pa)]:
        mu=c.mean();sd=c.std()+1e-8; b,lo,hi=auc_ci(np.abs((c-mu)/sd),np.abs((a-mu)/sd))
        print(f"  {nm:<16}{b:.3f} [{lo:.3f}, {hi:.3f}]")
print('\n표 4의 CI 및 AutoAttack 열 채움 완료.')